<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/kvcachemanager_scheduler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from collections import deque, OrderedDict
from dataclasses import dataclass, field
from enum import Enum
import math

class Policy(Enum):
  PREFILL_FIRST = "prefill_first"
  DECODE_FIRST="decode_first"

@dataclass
class Request:
  request_id: int
  prompt_tokens: int
  max_new_tokens: int
  generated_tokens: int = 0
  prefilled: bool = False
  blocks: list[int] = field(default_factory=list)

@property
def total_tokens_needed(self):
  return self.prompt_tokens + self.max_new_tokens

@property
def finished(self):
  return self.generated_tokens >= self.max_new_tokens

class CacheManager:
  def __init__(self, total_blocks=10, block_size=4):
    self.free_blocks = deque(range(total_blocks))
    self.active_blocks = {}
    self.cached_blocks = OrderedDict() #stores finished request blocks for possible reuse

  def blocks_needed(self, num_tokens):
    return math.ceil(num_tokens / self.block_size) #if block size = 4 and there are 12 tokens - 12/4 - 3 blocks needed

  def allocate(self, request: Request, tokens_needed: int):
    needed = self.blocks_needed(tokens_needed)
    while len(self.free_blocks) < needed:
        self.evict_lru()

    blocks = [self.free_blocks.popleft() for _ in range(needed)]
    self.active_blocks[request.request_id] = blocks
    request.blocks = blocks
    print(f"Allocated blocks {blocks} to request {request.request_id}")

  def can_allocate(self, tokens_needed):
    needed = self.blocks_needed(tokens_needed)
    return needed <= self.total_blocks

  def free_active(self, request: Request, keep_cached=True):
    blocks = self.active_blocks.pop(request.request_id, [])

    if keep_cached:
      self.cached_blocks[request.request_id] = blocks
      self.cached_blocks.move_to_end(request.request_id)
      print(f"Moved request {request.request_id} to cached blocks {blocks}")
    else:
      self.free_blocks.extend(blocks)
      print(f"Freed blocks {request.request_id} blocks: {blocks}")

  def reuse_cache(self, request: Request):
    if request.request_id in self.cached_blocks:
      blocks = self.cached_blocks.pop(request.request_id)
      self.active_blocks[request.request_id] = blocks
      request.blocks = blocks
      print(f"Reused cached blocks for request {request.request_id}: {blocks}")
      return True
    return False

  def evict_lru(self):
    if not self.cached_blocks:
      raise RuntimeError("No cached blocks to evict. Need preemption.")
    old_request_id, blocks = self.cached_blocks.popitem(last=False)
    self.free_blocks.extend(blocks)
    print(f"Evicted cached request {old_request_id}, freed blocks {blocks}")

In [ ]:
from collections import deque, OrderedDict
from dataclasses import dataclass, field
from enum import Enum
import math


class Policy(Enum):
    PREFILL_FIRST = "prefill_first"
    DECODE_FIRST = "decode_first"


@dataclass
class Request:
    request_id: int
    prompt_tokens: int
    max_new_tokens: int
    generated_tokens: int = 0
    prefilled: bool = False
    blocks: list[int] = field(default_factory=list)

    @property
    def total_tokens_needed(self):
        return self.prompt_tokens + self.max_new_tokens

    @property
    def finished(self):
        return self.generated_tokens >= self.max_new_tokens


class CacheManager:
    def __init__(self, total_blocks=10, block_size=4):
        self.total_blocks = total_blocks
        self.block_size = block_size
        self.free_blocks = deque(range(total_blocks))
        self.active_blocks = {}
        self.cached_blocks = OrderedDict()
    def blocks_needed(self, num_tokens):
        return math.ceil(num_tokens / self.block_size)

    def available_blocks(self):
        return len(self.free_blocks)

    def allocate(self, request: Request, tokens_needed: int):
        needed = self.blocks_needed(tokens_needed)

        while len(self.free_blocks) < needed:
            self.evict_lru()

        blocks = [self.free_blocks.popleft() for _ in range(needed)]
        self.active_blocks[request.request_id] = blocks
        request.blocks = blocks

        print(f"Allocated blocks {blocks} to request {request.request_id}")

    def can_allocate(self, tokens_needed):
        needed = self.blocks_needed(tokens_needed)
        return needed <= self.total_blocks

    def free_active(self, request: Request, keep_cached=True):
        blocks = self.active_blocks.pop(request.request_id, [])

        if keep_cached:
            self.cached_blocks[request.request_id] = blocks
            self.cached_blocks.move_to_end(request.request_id)
            print(f"Moved request {request.request_id} blocks to cache: {blocks}")
        else:
            self.free_blocks.extend(blocks)
            print(f"Freed request {request.request_id} blocks: {blocks}")

    def reuse_cache(self, request: Request):
        if request.request_id in self.cached_blocks:
            blocks = self.cached_blocks.pop(request.request_id)
            self.active_blocks[request.request_id] = blocks
            request.blocks = blocks
            print(f"Reused cached blocks for request {request.request_id}: {blocks}")
            return True
        return False

    def evict_lru(self):
        if not self.cached_blocks:
            raise RuntimeError("No cached blocks to evict. Need preemption.")

        old_request_id, blocks = self.cached_blocks.popitem(last=False)
        self.free_blocks.extend(blocks)
        print(f"Evicted cached request {old_request_id}, freed blocks {blocks}")


class Scheduler:
    def __init__(self, cache: CacheManager, policy=Policy.PREFILL_FIRST, max_batch_size=3):
        self.cache = cache
        self.policy = policy
        self.max_batch_size = max_batch_size
        self.waiting = deque()
        self.running = []

    def submit(self, request: Request):
        print(f"Submitted request {request.request_id}")
        self.waiting.append(request)

    def admit_new_requests(self):
        while self.waiting and len(self.running) < self.max_batch_size:
            req = self.waiting[0]
            tokens_needed = req.total_tokens_needed

            if not self.cache.can_allocate(tokens_needed):
                print(f"Request {req.request_id} too large for cache")
                self.waiting.popleft()
                continue

            try:
                self.waiting.popleft()

                if not self.cache.reuse_cache(req):
                    self.cache.allocate(req, tokens_needed)

                self.running.append(req)

            except RuntimeError:
                print("Memory full. Cannot admit more requests now.")
                break

    def prefill_step(self):
        for req in self.running:
            if not req.prefilled:
                req.prefilled = True
                print(f"Prefilled request {req.request_id}")

    def decode_step(self):
        for req in self.running:
            if req.prefilled and not req.finished:
                req.generated_tokens += 1
                print(
                    f"Decoded token {req.generated_tokens}/"
                    f"{req.max_new_tokens} for request {req.request_id}"
                )

    def cleanup_finished(self):
        still_running = []

        for req in self.running:
            if req.finished:
                print(f"Request {req.request_id} finished")
                self.cache.free_active(req, keep_cached=True)
            else:
                still_running.append(req)

        self.running = still_running

    def step(self):
        print("\n--- Scheduler step ---")

        if self.policy == Policy.PREFILL_FIRST:
            self.admit_new_requests()
            self.prefill_step()
            self.decode_step()

        elif self.policy == Policy.DECODE_FIRST:
            self.decode_step()
            self.cleanup_finished()
            self.admit_new_requests()
            self.prefill_step()

        self.cleanup_finished()

        print(f"Free blocks: {list(self.cache.free_blocks)}")
        print(f"Cached blocks: {dict(self.cache.cached_blocks)}")


if __name__ == "__main__":
    cache = CacheManager(total_blocks=8, block_size=4)

    scheduler = Scheduler(
        cache=cache,
        policy=Policy.PREFILL_FIRST,
        max_batch_size=2
    )

    scheduler.submit(Request(request_id=1, prompt_tokens=8, max_new_tokens=4))
    scheduler.submit(Request(request_id=2, prompt_tokens=6, max_new_tokens=3))
    scheduler.submit(Request(request_id=3, prompt_tokens=4, max_new_tokens=2))

    for _ in range(8):
        scheduler.step()

Submitted request 1
Submitted request 2
Submitted request 3

--- Scheduler step ---
Allocated blocks [0, 1, 2] to request 1
Allocated blocks [3, 4, 5] to request 2
Prefilled request 1
Prefilled request 2
Decoded token 1/4 for request 1
Decoded token 1/3 for request 2
Free blocks: [6, 7]
Cached blocks: {}

--- Scheduler step ---
Decoded token 2/4 for request 1
Decoded token 2/3 for request 2
Free blocks: [6, 7]
Cached blocks: {}

--- Scheduler step ---
Decoded token 3/4 for request 1
Decoded token 3/3 for request 2
Request 2 finished
Moved request 2 blocks to cache: [3, 4, 5]
Free blocks: [6, 7]
Cached blocks: {2: [3, 4, 5]}

--- Scheduler step ---
Allocated blocks [6, 7] to request 3
Prefilled request 3
Decoded token 4/4 for request 1
Decoded token 1/2 for request 3
Request 1 finished
Moved request 1 blocks to cache: [0, 1, 2]
Free blocks: []
Cached blocks: {2: [3, 4, 5], 1: [0, 1, 2]}

--- Scheduler step ---
Decoded token 2/2 for request 3
Request 3 finished
Moved request 3 blocks to 